# DS2002 · ETL Concepts and Walmart Case Framing

**Lecture — 2026-10-12 · Fall 2026**

---

## Everything so far is one pipeline

Everything you've done this semester fits into three stages. You've been doing all three already, just without the names.

| Stage | What it means | What you already know |
|---|---|---|
| **Extract** | Get the data out of wherever it lives | `read_csv`, `read_sql_query`, `requests.get` with retries |
| **Transform** | Make it correct, consistent, and joined | dedupe, coerce types, normalize text, merge, validate |
| **Load** | Put the result somewhere the next person can use | `to_sql`, `to_csv`, a clean table with a documented schema |

The midterm and the capstone are both ETL pipelines. Today is about what makes a pipeline reliable enough to run every week, as opposed to a notebook that worked once.

### Four things a good pipeline does

**1. It gives the same answer every time.** The fancy word is idempotent. Run it twice and you get the same result. This is the habit from week two: never change the source, always make a new copy.

**2. It checks its work.** Check row counts and add assertions after every extract and every join. If something breaks, you want to see it right there, not three cells later in a chart.

**3. It fails loudly.** No bare `except`. If a pipeline returns an empty DataFrame when something goes wrong, that's worse than a crash, because someone will read the empty result as an answer.

**4. It writes down its decisions.** Which rows you dropped, which SKUs you merged, what you assumed. Without that, nobody can check your work.

These four are the rubric for both projects, so keep them in mind for the rest of class.

### The Walmart case

Before Hurricane Frances hit in 2004, Walmart's CIO went through years of point-of-sale data from earlier storms. The question was simple: what do people actually buy when a hurricane is coming?

The expected items showed up: flashlights, bottled water. The surprise was **strawberry Pop-Tarts, selling at about seven times their normal rate** before a storm, with beer close behind. Nobody guessed that. It came out of the data.

Read `Walmart Case.pdf` in this folder. We'll read it in class and then break into groups. It sets up your **midterm**, which kicks off next Monday.

Two things to keep in mind as you read:

- The finding only worked because the data was **consolidated**. Pop-Tarts show up under many SKUs and spellings across thousands of stores. Split up like that, you can't see the signal.
- It was only *useful* because someone acted on it: trucks rerouted, shelves restocked before landfall. An analysis that doesn't change a decision isn't worth much.

### Read the case, then discuss

Take about 15 minutes to read `Walmart Case.pdf`, including the exhibits. Then get into groups and work through these. Use the exhibits, not just the story.

1. **The SKU call (Exhibit 1).** `PT-12`, `SKU#459812`, and `POPTART-STRAWBERRY` are all strawberry Pop-Tarts, but the junior analyst isn't sure `PT-12` and `SKU#459812` are the same product. What would you check before merging them? What does a wrong merge cost, compared with leaving them split?

2. **Percent or units (Exhibit 2).** Pop-Tarts went up 600% and bottled water went up 300%. Work out how many extra units per day each one adds. Which number would you give the operations chief to decide what goes on the trucks, and why?

3. **Time zones and the missing timestamp (Exhibits 1 and 3).** The weather feed is in GMT and the sales data is in EST. `TXN006` has no timestamp at all. How would your sales-to-weather join go wrong if you ignored the time zone? For the NULL row, would you drop it, fill it in, or flag it, and how might that choice change the answer?

4. **The board game bet (Exhibit 6).** Board games are up 250%, playing cards up 175%, and streaming devices up 180%. The Finance VP thinks Monopoly and "card game bundles" got lumped into one category. You have 48 hours. What would you check in the data to decide whether the signal is real? What would you need to see before shipping product?

5. **Whose argument wins.** The Chief Data Officer says the risk is the pipeline, not the model: "We need cleaner wrangling before fancier algorithms." Do you agree? Pick one of the four pipeline properties above and explain how it would have caught a bad Pop-Tarts number before anyone shipped a truck.

### Consolidation on a small scale

Here's a small version of how split-up SKUs hide a real signal. Four transactions, one product, and one row that got scanned twice.

In [ ]:
import pandas as pd

rows = pd.DataFrame({
    'store': ['FL-105', 'FL-239', 'FL-105', 'FL-105'],
    'sku': ['PT-12', 'SKU#459812', 'POPTART-STRAW', 'PT-12'],
    'name': ['pop-tarts strawberry', 'Strawberry Pop-Tart',
             'Pop-Tarts Strawberry', 'pop-tarts strawberry'],
    'qty': [40, 15, 22, 40],
})
rows

In [ ]:
# The fragmented view: looks like three unremarkable products.
# The double-scanned row is dropped first so we are comparing like with like.
deduped = rows.drop_duplicates()
print(deduped.groupby('sku')['qty'].sum().sort_values(ascending=False))
print()
print('largest single SKU:', deduped.groupby('sku')['qty'].sum().max())

Forty units. Against a normal week, that's noise, and nobody reroutes a truck for it.

Now consolidate. The order matters here, and getting it backwards is a classic mistake. Deduplicate **first**. If you map to a canonical SKU before you drop duplicates, different rows can end up looking identical, and you can't tell a double scan from two real sales.

In [ ]:
CANON = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW',
}

clean = rows.drop_duplicates()                    # dedupe FIRST
clean = clean.assign(canonical=clean['sku'].map(CANON))
print('rows:', len(rows), '->', len(clean))
print()
print(clean.groupby('canonical')['qty'].sum())

In [ ]:
fragmented = deduped.groupby('sku')['qty'].sum().max()
consolidated = clean.groupby('canonical')['qty'].sum().max()
print(f'biggest number you can see fragmented:   {fragmented}')
print(f'biggest number after consolidation:      {consolidated}')
print(f'the signal was {consolidated / fragmented:.2f}x larger than it looked')

Seventy-seven units, not forty. Same data. The only difference is that we recognized it was one product.

**Discussion (2 minutes, with the person next to you):** When does this stop being data cleaning and turn into a business decision? `PT-12` and `POPTART-STRAW` are clearly the same product. What about strawberry vs. blueberry Pop-Tarts, one product or two? What about a 6-pack vs. a 12-pack of the same flavor?

There's no single right answer. The mapping table is where you write down your choice so someone else can see it and disagree.

### Where the mapping comes from

With thousands of products, you can't type the dictionary by hand. Instead, you build a **candidate list** with code, then a person confirms it.

This is the workflow for the midterm: use code to narrow it down, then make the calls yourself.

In [ ]:
catalog = pd.DataFrame({
    'sku': ['PT-12', 'SKU#459812', 'POPTART-STRAW', 'PT-BLUE-6',
            'WATER-24', 'BTL-WATER-24', 'FLASH-AA'],
    'name': ['pop-tarts strawberry', 'Strawberry Pop-Tart',
             'Pop-Tarts Strawberry', 'Pop-Tarts Blueberry 6ct',
             'Bottled Water 24pk', 'bottled water 24 pack', 'Flashlight AA'],
})

# Use code to narrow it down: group by a normalized name and look for matches
catalog['normalized'] = (catalog['name'].str.lower()
                         .str.replace(r'[^a-z ]', '', regex=True)
                         .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (catalog.groupby('normalized')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])
candidates

Lowercasing, stripping punctuation, and sorting the words matches `PT-12` and `POPTART-STRAW`, since their names only differ in case and word order. It leaves blueberry alone, which is right. It misses `Strawberry Pop-Tart` (`tart` vs. `tarts`) and the two bottled water SKUs (`pk` vs. `pack`).

No automated rule gets all of it. What the code does is cut thousands of SKUs down to a short list of judgment calls, and then a person makes those calls.

### Load: the step people skip

Extract and transform get most of the attention, but load is what makes the work reusable. The clean table feeds every question that comes after it, and nobody has to re-run your cleaning to use it.

In this course, loading means writing the clean table to SQLite and querying it back to check that it's there. You'll do this yourself on Wednesday.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
out = clean[['store', 'canonical', 'qty']]
out.to_sql('clean_sales', conn, index=False, if_exists='replace')

# Read it back to check the load worked
check = pd.read_sql_query(
    'SELECT canonical, SUM(qty) AS units FROM clean_sales GROUP BY canonical', conn)
print(check)
print()
print('rows written:', pd.read_sql_query(
    'SELECT COUNT(*) AS n FROM clean_sales', conn)['n'][0], 'of', len(out))

### What's coming

This week, Wednesday's studio is a dress rehearsal: a full extract, transform, and load on a small sample of the Walmart data. Friday's lab gets you into the real SKU catalog.

Next Monday is the midterm kickoff. You get the full dataset: hundreds of thousands of transactions, fragmented SKUs, three timestamp formats, negative quantities, and store ids in four different casings. You'll work in teams of two to four for about three weeks. The deliverable is a pipeline, five answered questions, and a recommendation.

The teams that do well understand the business question before they start writing pandas, so hold on to what you got out of the case today.